In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from tqdm import tqdm
import os
import yaml
import zuko
from helpers.data_transforms import preprocess_data, inverse_preprocess_data, load_in_data

from helpers.models.DNN import count_parameters
from helpers.evaluation import run_eval_suite_BDTs, run_eval_suite_R
from helpers.plotting import plot_hists_1d, plot_corner_hist_2d, make_2d_plots
from helpers.material_map import apply_material_map_hybrid


plt.style.use("../science.mplstyle")

In [9]:
BIN_BOUND = 5
NUM_BINS = 60
NUM_FEATURES = 5
NUM_BDTS = 10

collections = [
#    "InnerTrackerBarrelCollection",
  # "InnerTrackerEndcapCollection",
 #   "OuterTrackerBarrelCollection",     
    "OuterTrackerEndcapCollection",  
 #   "VertexBarrelCollection",   
#  "VertexEndcapCollection",
]


feature_order_endcap = [0,4,1,2,3,5,6,7,8]
feature_indices_dict_endcap = {
    "r":2,
    "phi": 3,
    "z":4,
    "side":5,
    "layer":6
}

feature_order_barrel =[0, 4, 2, 3, 1,  5, 6]
feature_indices_dict_barrel = {
    "r":4,
    "phi": 2,
    "z":3,
    "side":5,
    "layer":6
}

feature_order_dict = {
     "InnerTrackerBarrelCollection":feature_order_barrel,
    "InnerTrackerEndcapCollection":feature_order_endcap,
    "OuterTrackerBarrelCollection":feature_order_barrel,
    "OuterTrackerEndcapCollection":feature_order_endcap,  
    "VertexBarrelCollection":feature_order_barrel,
    "VertexEndcapCollection":feature_order_endcap,

}


feature_indices_dict = {
     "InnerTrackerBarrelCollection":feature_indices_dict_barrel,
    "InnerTrackerEndcapCollection":feature_indices_dict_endcap,
    "OuterTrackerBarrelCollection":feature_indices_dict_barrel,
    "OuterTrackerEndcapCollection":feature_indices_dict_endcap,  
    "VertexBarrelCollection":feature_indices_dict_barrel,
    "VertexEndcapCollection":feature_indices_dict_endcap,

}



ZUKO_ID = "NCSF"
NAME = "cond3"
NUM_COND_INPUTS = 3
SEED = 8



FEATURES = "rphi"
working_dir = "/pscratch/sd/r/rmastand/muon_collider"
log_vars = []



In [10]:
# load in samples

all_data_dir, all_samples_dir = {}, {}
bins_dict, bins_dict_preproc = {}, {}
aucs = {}
feature_labels_dict = {}

for col_name in collections:

    small_id = ''.join([c for c in col_name if c.isupper()])
    X, feature_labels = load_in_data([col_name], FEATURES, working_dir, 1, NUM_COND_INPUTS, feature_order_dict[col_name])
    
    all_data_dir[col_name] = X
    all_samples_dir[col_name] = np.load(f"{working_dir}/zuko_outputs/{ZUKO_ID}/{small_id}_{NAME}/flow_samples.npy")
    #with open(f"{working_dir}/zuko_outputs/{ZUKO_ID}/{small_id}_{NAME}/results.txt") as ifile:
       # aucs[col_name]=  ifile.readlines()[-1]
    feature_labels_dict[col_name] = feature_labels



    # project from local phi -> phi

    num_sectors = {
    "InnerTrackerEndcapCollection": 26, 
    "OuterTrackerEndcapCollection": 48, 
    "VertexEndcapCollection": 16,
    }

    # baseline = np.random.choice(range(num_sectors[col_name]), shape = len(all_samples_dir[col_name])) 
    # print(baseline)
    # baseline = 2*np.pi* baseline / range(num_sectors[col_name]
    # print(baseline)

    # if FEATURES == "xy":

    #     r_data = np.sqrt(all_data_dir[col_name][:,2]**2 + all_data_dir[col_name][:,3]**2)
    #     phi_data = np.arctan2(all_data_dir[col_name][:,3], all_data_dir[col_name][:,2])

    #     r_samples = np.sqrt(all_samples_dir[col_name][:,2]**2 + all_samples_dir[col_name][:,3]**2)
    #     phi_samples = np.arctan2(all_samples_dir[col_name][:,3], all_samples_dir[col_name][:,2])

    #     if "Barrel" in col_name:
    #         # e t phi z r
    #         all_data_dir[col_name][:,2] = phi_data
    #         all_data_dir[col_name][:,3] = all_data_dir[col_name][:,4]
    #         all_data_dir[col_name][:,4] = r_data
    #         feature_labels_dict[col_name] = feature_labels = ["log($E$) [Gev]", "$t$ [s]", "$\phi$", "$z$",  "$r$", "side", "layer"]


    #         all_samples_dir[col_name][:,2] = phi_samples
    #         all_samples_dir[col_name][:,3] = all_samples_dir[col_name][:,4]
    #         all_samples_dir[col_name][:,4] = r_samples

    #     elif "Endcap" in col_name:
    #         # e t r phi z
    #         all_data_dir[col_name][:,2] = r_data
    #         all_data_dir[col_name][:,3] = phi_data
    #         feature_labels_dict[col_name] = feature_labels = ["log($E$) [Gev]", "$t$ [s]", "$r$", "$\phi$", "$z$", "side", "layer"]

    #         all_samples_dir[col_name][:,2] = r_samples
    #         all_samples_dir[col_name][:,3] = phi_samples

    # convert from log E, t, r, sinphi, cosphi, z, side, layer to log E, t, r, phi, z, side, layer
    

    
    

    bins_dict[col_name] = {}
    bins_dict_preproc[col_name] = {i:np.linspace(-BIN_BOUND, BIN_BOUND, NUM_BINS) for i in range(all_data_dir[col_name].shape[1])}
    
    for i in range(all_data_dir[col_name].shape[1]):
        if i in log_vars:
            bins_dict[col_name][i] = np.logspace(np.log10(0.9*np.min(all_data_dir[col_name][:,i])), np.log10(1.1*np.max(all_data_dir[col_name][:,i])), NUM_BINS) 
        else:
            bins_dict[col_name][i] = np.linspace(np.min(all_data_dir[col_name][:,i] - 3), np.max(all_data_dir[col_name][:,i] + 3), NUM_BINS) 

In [11]:
for key in aucs.keys():
    print(f"{key}: {aucs[key]}")    

In [ ]:
# snap flow samples



z_side_layer_map_endcaps = {'InnerTrackerEndcapCollection': {(1, 0): {'starts': [522.849143562149, 528.8496556051206], 'stops': [522.8533456013948, 528.8538576443664]}, (1, 1): {'starts': [803.1461424545209, 809.1466505538285], 'stops': [803.1503444910051, 809.1508525903126]}, (1, 2): {'starts': [1091.8491427589308, 1097.8496554190353], 'stops': [1091.8533447986088, 1097.8538574587133]}, (1, 3): {'starts': [1372.1461428038197, 1378.1466468450215], 'stops': [1372.150344837462, 1378.1508488786637]}, (1, 4): {'starts': [1659.849143382624, 1665.8496551612313], 'stops': [1659.8533454216847, 1665.853857200292]}, (1, 5): {'starts': [1941.14614220122, 1947.1466522540668], 'stops': [1941.1503442390722, 1947.150854291919]}, (1, 6): {'starts': [2188.8491463031573, 2194.849654804879], 'stops': [2188.8533483399233, 2194.8538568416448]}, (-1, 0): {'starts': [-528.8538575003843, -522.8533441678617], 'stops': [-528.8496554602355, -522.8491421277129]}, (-1, 1): {'starts': [-809.1508543740399, -803.1503441588799], 'stops': [-809.1466523360741, -803.1461421209141]}, (-1, 2): {'starts': [-1097.8538571301856, -1091.8533453594218], 'stops': [-1097.8496550911304, -1091.8491433203667]}, (-1, 3): {'starts': [-1378.1508541816436, -1372.1503459821215], 'stops': [-1378.1466521450893, -1372.1461439455672]}, (-1, 4): {'starts': [-1665.8538513466156, -1659.8533491809549], 'stops': [-1665.8496493142868, -1659.8491471486261]}, (-1, 5): {'starts': [-1947.1508578598198, -1941.1503491779981], 'stops': [-1947.1466558229276, -1941.146147141106]}, (-1, 6): {'starts': [-2194.853854018727, -2188.8533447803366], 'stops': [-2194.8496519814453, -2188.849142743055]}}, 'OuterTrackerEndcapCollection': {(1, 0): {'starts': [1305.1461421633815, 1311.1466553002076], 'stops': [1305.1503442033934, 1311.1508573402195]}, (1, 1): {'starts': [1615.849141970229, 1621.8496537024448], 'stops': [1615.8533440092572, 1621.853855741473]}, (1, 2): {'starts': [1878.1461429805634, 1884.1466555550721], 'stops': [1878.1503450201815, 1884.1508575946903]}, (1, 3): {'starts': [2188.849143000882, 2194.849655625588], 'stops': [2188.8533450405357, 2194.853857665242]}, (-1, 0): {'starts': [-1311.1508565225504, -1305.1503443189094], 'stops': [-1311.1466544831922, -1305.1461422795512]}, (-1, 1): {'starts': [-1621.8538551980234, -1615.8533445284986], 'stops': [-1621.8496531597395, -1615.8491424902147]}, (-1, 2): {'starts': [-1884.1508569963762, -1878.1503446670397], 'stops': [-1884.14665495693, -1878.1461426275935]}, (-1, 3): {'starts': [-2194.8538577922936, -2188.8533446672172], 'stops': [-2194.8496557522903, -2188.849142627214]}}, 'VertexEndcapCollection': {(1, 0): {'starts': [80.13889753902934], 'stops': [80.14110237939136]}, (1, 1): {'starts': [84.188897861894], 'stops': [84.19110270146345]}, (1, 2): {'starts': [120.13889763560879], 'stops': [120.14110247612658]}, (1, 3): {'starts': [124.18889759575274], 'stops': [124.19110243659355]}, (1, 4): {'starts': [200.13889748444515], 'stops': [200.14110232480758]}, (1, 5): {'starts': [204.18889767615738], 'stops': [204.19110251665663]}, (1, 6): {'starts': [280.13889765423824], 'stops': [280.1411024943865]}, (1, 7): {'starts': [284.18889776666776], 'stops': [284.1911026063793]}, (-1, 0): {'starts': [-80.14110237190039], 'stops': [-80.13889753183304]}, (-1, 1): {'starts': [-84.19110240027032], 'stops': [-84.1888975596242]}, (-1, 2): {'starts': [-120.1411023063737], 'stops': [-120.13889746609716]}, (-1, 3): {'starts': [-124.19110254374714], 'stops': [-124.1888977039106]}, (-1, 4): {'starts': [-200.14110244532807], 'stops': [-200.1388976055517]}, (-1, 5): {'starts': [-204.1911026621207], 'stops': [-204.18889782292106]}, (-1, 6): {'starts': [-280.1411018246637], 'stops': [-280.1388969864688]}, (-1, 7): {'starts': [-284.19110222185054], 'stops': [-284.18889738388157]}}}
def snap_z_to_detector(z_vals, sides, layers, z_dict):
    z_new = np.copy(z_vals)

    for i in range(len(z_vals)):

        z = z_vals[i]
        side = sides[i]
        layer = layers[i]

        
        key = (side, layer)
        if key not in z_dict:
            print("error")

        starts = np.array(z_dict[key]["starts"])
        stops = np.array(z_dict[key]["stops"])

        # distance to each interval
        # 0 if inside, otherwise distance to nearest edge
        dist_to_intervals = np.where(
            (z >= starts) & (z <= stops),
            0,
            np.minimum(np.abs(z - starts), np.abs(z - stops))
        )
        # pick closest interval
        idx = np.argmin(dist_to_intervals)

        # sample uniformly inside that interval
        z_new[i] = np.random.uniform(starts[idx], stops[idx])

    return z_new


for col_name in collections:
    plt.figure()

    z_idx = feature_indices_dict[col_name]["z"]
    side_idx = feature_indices_dict[col_name]["side"]
    layer_idx = feature_indices_dict[col_name]["layer"]

    # original samples
    z_samples = all_samples_dir[col_name][:, z_idx]
    sides = all_samples_dir[col_name][:, side_idx]
    sides = [1 if s > 0 else -1 for s in sides]
    layers = all_samples_dir[col_name][:, layer_idx]

    z_definitions_dict = z_side_layer_map_endcaps[col_name]

    # snap z to detector geometry
    z_samples_snapped = snap_z_to_detector(
        z_samples, sides, layers, z_definitions_dict
    )
    all_samples_dir[col_name][:,feature_indices_dict[col_name]["z"]] = z_samples_snapped

    # plot truth vs snapped samples
    plt.hist(
        all_data_dir[col_name][:, z_idx],
        bins=np.linspace(-2000, 2000, 1000),
        histtype="step",
        label="truth"
    )

    plt.hist(
        z_samples_snapped,
        bins=np.linspace(-2000, 2000, 1000),
        histtype="step",
        label="flow (snapped)"
    )

    plt.yscale("log")
    plt.legend()
    plt.show()

In [ ]:
# for col_name in collections:

#     print(col_name)

#     plot_hists_1d(
#         {"data": all_data_dir[col_name], "samples": all_samples_dir[col_name]},
#         bins_dict[col_name],
#         log_dims=log_vars,
#         labels=feature_labels_dict[col_name]
#     )
#     plt.savefig(f"figures/{col_name}_1d.png")

#     loc_array = {
#         "data": (
#             all_data_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
#             np.cos(all_data_dir[col_name][:, feature_indices_dict[col_name]["phi"]]),
#             all_data_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
#             np.sin(all_data_dir[col_name][:, feature_indices_dict[col_name]["phi"]])
#         ),
#         "samples": (
#             all_samples_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
#             np.cos(all_samples_dir[col_name][:, feature_indices_dict[col_name]["phi"]]),
#             all_samples_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
#             np.sin(all_samples_dir[col_name][:, feature_indices_dict[col_name]["phi"]])
#         )
#     }

#     make_2d_plots(loc_array, ["x", "y"])

#     loc_array = {
#         "data": (
#             all_data_dir[col_name][:, feature_indices_dict[col_name]["r"]],
#             all_data_dir[col_name][:, feature_indices_dict[col_name]["z"]]
#         ),
#         "samples": (
#             all_samples_dir[col_name][:, feature_indices_dict[col_name]["r"]],
#             all_samples_dir[col_name][:, feature_indices_dict[col_name]["z"]]
#         )
#     }

#     make_2d_plots(loc_array, ["r", "z"])

In [ ]:
"""
for col_name in collections:
    fig_samp, axes_samp = plot_corner_hist_2d(
       all_samples_dir[col_name],
        feature_labels=feature_labels,
        bins_dict=bins_dict[col_name],
        log_dims=log_vars,
        title= "flow_samples",
    )
    plt.savefig(f"figures/{col_name}_2d.png")
"""


In [ ]:

for col_name in collections:

    print(col_name)
    mask = apply_material_map_hybrid(all_samples_dir, None, col_name, feature_indices_dict)
    
    print(f"{100*sum(mask)/len(mask)}% of samples pass ({sum(mask)}, {len(mask)})")
    plot_hists_1d({
        "data":all_data_dir[col_name], 
         "masked samples": all_samples_dir[col_name][mask],
        "unmasked samples":  all_samples_dir[col_name]
    }, bins_dict[col_name], log_dims=log_vars, labels=feature_labels)
    plt.savefig(f"figures/{col_name}_1d.png")


    
    loc_array = {
        "data": (
            all_data_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
            np.cos(all_data_dir[col_name][:, feature_indices_dict[col_name]["phi"]]),
            all_data_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
            np.sin(all_data_dir[col_name][:, feature_indices_dict[col_name]["phi"]])
        ),
        "samples": (
            all_samples_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
            np.cos(all_samples_dir[col_name][:, feature_indices_dict[col_name]["phi"]]),
            all_samples_dir[col_name][:, feature_indices_dict[col_name]["r"]] *
            np.sin(all_samples_dir[col_name][:, feature_indices_dict[col_name]["phi"]])
        ),
        "masked samples": (
            all_samples_dir[col_name][mask][:, feature_indices_dict[col_name]["r"]] *
            np.cos(all_samples_dir[col_name][mask][:, feature_indices_dict[col_name]["phi"]]),
            all_samples_dir[col_name][mask][:, feature_indices_dict[col_name]["r"]] *
            np.sin(all_samples_dir[col_name][mask][:, feature_indices_dict[col_name]["phi"]])
        )
    }

    make_2d_plots(loc_array, ["x", "y"])

    loc_array = {
        "data": (
            all_data_dir[col_name][:, feature_indices_dict[col_name]["r"]],
            all_data_dir[col_name][:, feature_indices_dict[col_name]["z"]]
        ),
        "samples": (
            all_samples_dir[col_name][:, feature_indices_dict[col_name]["r"]],
            all_samples_dir[col_name][:, feature_indices_dict[col_name]["z"]]
        ),
        "masked samples": (
            all_samples_dir[col_name][mask][:, feature_indices_dict[col_name]["r"]],
            all_samples_dir[col_name][mask][:, feature_indices_dict[col_name]["z"]]
        )
    }

    make_2d_plots(loc_array, ["r", "z"])

In [ ]:
single_bdt_results, all_scores, all_plot = {}, {}, {}

import torch
device = torch.device( "cuda" if torch.cuda.is_available() else "cpu")
print( "Using device: " + str( device ), flush=True)
num_BDTs = 1


for col_name in collections:

    print(f"Analyzing {col_name}...")

    
    mask = apply_material_map_hybrid(all_samples_dir, None, col_name, feature_indices_dict)
    N = 1_000_000
    print(np.sum(mask))



    # if "Barrel" in col_name:
    #     # e t phi z r to e t x y z
    
    #     data_x = all_data_dir[col_name][:,4]* np.cos(all_data_dir[col_name][:,2])
    #     data_y = all_data_dir[col_name][:,4]* np.sin(all_data_dir[col_name][:,2])
    
    #     all_data_dir[col_name][:,4] = all_data_dir[col_name][:,3]
    #     all_data_dir[col_name][:,2] = data_x
    #     all_data_dir[col_name][:,3] = data_y
        
    
    #     samples_x = all_samples_dir[col_name][:,4]* np.cos(all_samples_dir[col_name][:,2])
    #     samples_y = all_samples_dir[col_name][:,4]* np.sin(all_samples_dir[col_name][:,2])
        
    
    
    #     all_samples_dir[col_name][:,4] = all_samples_dir[col_name][:,3]
    #     all_samples_dir[col_name][:,2] = samples_x
    #     all_samples_dir[col_name][:,3] = samples_y
    
    # elif "Endcap" in col_name:
    #     # e t r phi z  to e t x y z
    
    #     data_x = all_data_dir[col_name][:,2]* np.cos(all_data_dir[col_name][:,3])
    #     data_y = all_data_dir[col_name][:,2]* np.sin(all_data_dir[col_name][:,3])
    
        
    #     all_data_dir[col_name][:,2] = data_x
    #     all_data_dir[col_name][:,3] = data_y
    
    #     data_x = all_samples_dir[col_name][:,2]* np.cos(all_samples_dir[col_name][:,3])
    #     data_y = all_samples_dir[col_name][:,2]* np.sin(all_samples_dir[col_name][:,3])
    
        
    #     all_samples_dir[col_name][:,2] = data_x
    #     all_samples_dir[col_name][:,3] = data_y
    
    

     
    for i in range(all_data_dir[col_name].shape[1]):
        if i in log_vars:
            bins_dict[col_name][i] = np.logspace(np.log10(0.9*np.min(all_data_dir[col_name][:,i])), np.log10(1.1*np.max(all_data_dir[col_name][:,i])), NUM_BINS) 
        else:
            bins_dict[col_name][i] = np.linspace(np.min(all_data_dir[col_name][:,i] - 3), np.max(all_data_dir[col_name][:,i] + 3), NUM_BINS) 
    
    indices_1 = np.random.choice(all_data_dir[col_name].shape[0], size=N, replace=False)
    indices_2 = np.random.choice(all_samples_dir[col_name].shape[0], size=N, replace=False)
    

    print(all_data_dir[col_name].shape, all_samples_dir[col_name].shape, mask.shape)

    #indices_3 = np.random.choice(all_samples_dir[col_name][mask].shape[0], size=N, replace=False)

    tmp = run_eval_suite_BDTs(all_data_dir[col_name][indices_1], 
                              {"unmasked":all_samples_dir[col_name][indices_2], 
                               #"masked":all_samples_dir[col_name][mask][indices_3]
                              },
                                bins_dict[col_name],
                                device=device,
                                num_BDTs=num_BDTs,
                                plot_suffix=f"_{col_name}")

    single_bdt_results[col_name] = tmp[0]
    all_scores[col_name] = tmp[1]
    all_plot[col_name] = tmp[2]


  #  run_eval_suite_R(all_data_dir[col_name][indices_1], {"unmasked":all_samples_dir[col_name][indices_2], "masked":all_samples_dir[col_name][mask][indices_3]}, [0.1, 0.2, 0.4],  100, plot_suffix="")
#
    


In [ ]:

plt.figure()
for i, col_name in enumerate(single_bdt_results.keys()):
    if "Endcap" in col_name:
        for key in single_bdt_results[col_name].keys():
            x_vals = single_bdt_results[col_name][key].keys()
            
            res = [single_bdt_results[col_name][key][x][0] for x in x_vals]
            res_gauss = [single_bdt_results[col_name][key][x][1] for x in x_vals]
            if key == "masked":
                plt.scatter(x_vals, res, color = f"C{i}", label = f"{col_name}, masked", marker="o", s=100)
            elif key == "unmasked":
                plt.scatter(x_vals, res, edgecolor = f"C{i}", label = f"{col_name}, unmasked", marker="o", s=100, facecolor="none")
plt.legend(loc=(1,0))
plt.xticks([0, 1, 2, 3, 4, 5, 6, 7, 8], labels = feature_labels_dict[col_name], rotation = 45)
plt.xlabel("feature")

plt.ylabel("ROC AUC")
plt.show()

# Analyzing the features that didn't pass

In [ ]:
col_to_analyze = "InnerTrackerEndcapCollection"
percentile_to_analyze = 90




plt.figure(figsize = (10, 10))



passes =  np.array(all_scores[col_to_analyze]["masked"] >= np.percentile(all_scores[col_to_analyze]["unmasked"], percentile_to_analyze))

x = all_plot[col_to_analyze]["masked"][:,feature_indices_dict[col_to_analyze]["r"]]*np.cos(all_plot[col_to_analyze]["masked"][:,feature_indices_dict[col_to_analyze]["phi"]])
y = all_plot[col_to_analyze]["masked"][:,feature_indices_dict[col_to_analyze]["r"]]*np.sin(all_plot[col_to_analyze]["masked"][:,feature_indices_dict[col_to_analyze]["phi"]])

z = all_plot[col_to_analyze]["masked"][:,feature_indices_dict[col_to_analyze]["z"]]



plt.scatter(x[~passes], y[~passes], color = "green", s = 0.1, label = f"below {percentile_to_analyze} percentile")
#plt.scatter(x[passes], y[passes], color = "red", s = 0.1, label = f"above {percentile_to_analyze} percentile")

# plt.scatter(x[~passes][z[~passes] > 0], y[~passes][z[~passes]> 0], color = "green", s = 1, label = f"below {percentile_to_analyze} percentile")
# plt.scatter(x[passes][z[passes] > 0], y[passes][z[passes] > 0], color = "red", s = 1, label = f"above {percentile_to_analyze} percentile")


plt.title(f"red: above {percentile_to_analyze} percentile")
#plt.xlim(-20, 0)
#plt.ylim(-40, -20)

plt.show()



In [ ]:

bb = np.linspace(0, 1500, 1000)

plt.figure()
plt.hist(all_data_dir[col_name][:,feature_indices_dict[col_name]["z"]], bins = bb, density = True, label = "data", histtype = "stepfilled")


for p in [99]:
    #tmp_masked =  all_plot[col_name]["masked"][all_scores[col_name]["masked"] >= np.percentile(all_scores[col_name]["masked"], p)]
    tmp_unmasked =  all_plot[col_name]["unmasked"][all_scores[col_name]["unmasked"] >= np.percentile(all_scores[col_name]["unmasked"], p)]

    
    plt.hist(tmp_unmasked[:,feature_indices_dict[col_name]["z"]], bins = bb, histtype = "step", density = True, label = "unmasked")
    
   # plt.hist(tmp_masked[:,feature_indices_dict[col_name]["z"]], bins = bb, histtype = "step", density = True, label = "masked")
plt.legend()
plt.yscale("log")
plt.show()

In [ ]:
bins = 1000
for col_name in collections:

    print(col_name)

    plt.figure(figsize = (10, 10))

    plt.hist2d(all_data_dir[col_name][:,feature_indices_dict[col_name]["r"]]*np.cos(all_data_dir[col_name][:,feature_indices_dict[col_name]["phi"]]), 
               all_data_dir[col_name][:,feature_indices_dict[col_name]["r"]]*np.sin(all_data_dir[col_name][:,feature_indices_dict[col_name]["phi"]]), 
               bins=(bins,bins),
              norm="log",)
    plt.show()
   